# Tutorial 2 — Tokenization Deep Dive

**Course:** Text & Language Processing / LLM Practical Track
**Format:** Hands-on notebook (Colab-ready)
**Suggested duration:** 120–150 minutes
**Prerequisite:** Tutorial 0 (Hugging Face Ecosystem Tour)

Tutorial 0 treated the tokenizer as a black box inside `pipeline()`. This notebook opens that box.

By the end of this notebook you will be able to:

- explain *why* LLMs don't operate on raw characters or whole words, but on **subword tokens**;
- run the **Byte-Pair Encoding (BPE)** merge algorithm by hand on a tiny corpus, so the Hugging Face
  implementation stops feeling like magic;
- compare **BPE, WordPiece, and Unigram/SentencePiece** on the same sentence and explain what differs;
- explain **byte-level BPE** (GPT-2/GPT-4 style) and why it never produces an "unknown token";
- read every argument of `tokenizer(...)` — `padding`, `truncation`, `max_length`, `return_tensors`,
  `return_attention_mask` — and predict what a batch will look like before running the cell;
- **train your own tokenizer from scratch** with the `tokenizers` library and load it back with
  `AutoTokenizer`;
- use `apply_chat_template` correctly for instruction-tuned models;
- diagnose the tokenization bugs that most commonly break real LLM applications (leading spaces, digit
  splitting, tokenizer/model mismatch, multilingual cost blow-up).


## 0. Mental model: text is not what the model sees

A language model never sees the string `"unbelievable"`. It sees a sequence of integers, e.g.
`[403, 6667, 720]`. Three separate maps make that translation possible, and each one is a design choice:

```text
raw text  --tokenizer.tokenize-->  token strings  --tokenizer.convert_tokens_to_ids-->  integer ids
   |                                                                                        |
   |                                                                                        v
   |                                                                              model.embedding(ids)
   |                                                                                        |
   +----------------------------- tokenizer.decode(ids) <---------------- model output ids -+
```

The **tokenizer** is a separate, trained artifact from the model. It has its own vocabulary file(s) and
its own training algorithm, and it is normally trained **once** and then frozen for the lifetime of a
model family. This matters practically: a model checkpoint and its tokenizer must always be loaded as a
*pair* (`AutoTokenizer.from_pretrained("same-repo-as-the-model")`) — mixing tokenizers from different
models silently produces garbage, because the integer ids mean different things to different models.

There are three traditional families of tokenization, and understanding why the field converged on the
third one is the goal of the next two sections:

| Granularity | Example split of "unbelievable" | Vocabulary size | Main failure mode |
|---|---|---|---|
| **Word-level** | `["unbelievable"]` | Huge (100k+), still has gaps | Out-of-vocabulary (OOV) words |
| **Character-level** | `["u","n","b","e","l","i","e","v","a","b","l","e"]` | Tiny (~100) | Sequences become very long; hard to learn meaning per character |
| **Subword-level** | `["un", "believ", "able"]` | Moderate (16k–128k) | Occasional awkward splits, but no OOV and manageable sequence length |

Every modern LLM (BERT, GPT-2/3/4, Llama, T5, Qwen, ...) uses some form of **subword tokenization**. The
rest of this notebook builds that idea from first principles, then shows you the production tools.


## 1. Install the libraries

In [ ]:
!pip -q install -U transformers tokenizers datasets tiktoken


### Code walkthrough — installation command

```bash
!pip -q install -U transformers tokenizers datasets tiktoken
```

- **`transformers`**: provides `AutoTokenizer`, which loads any pretrained tokenizer (BPE, WordPiece, or
  Unigram) behind one uniform interface.
- **`tokenizers`**: the fast, Rust-backed library that `AutoTokenizer` uses under the hood, and which we
  will also use directly in Section 5 to **train** a tokenizer from scratch.
- **`datasets`**: supplies a small real corpus to train and evaluate tokenizers on.
- **`tiktoken`**: OpenAI's byte-level BPE tokenizer library, used in Section 4 to inspect GPT-4-style
  tokenization without needing a Hugging Face checkpoint.


## 2. Why not just split on whitespace?

The simplest possible tokenizer splits on spaces and punctuation. Let's measure what that choice costs
on real text before dismissing it — the failure modes are easier to trust once you've seen the numbers.


In [ ]:
import re
from collections import Counter

corpus = [
    "Tokenization is the first step of every NLP pipeline.",
    "Tokenisation is the first step of every NLP pipeline.",  # British spelling
    "The pre-tokenizer splits text before subword merges are applied.",
    "unbelievable, unbelievably, believable, believer, disbelief",
    "COVID-19 vaccination rates rose in 2021.",
    "She's running the tokenizer's unit tests at 3:45pm.",
]

def whitespace_tokenize(text):
    return text.split()

def word_punct_tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text)

for text in corpus:
    print(f"{text!r}")
    print("  whitespace :", whitespace_tokenize(text))
    print("  word+punct :", word_punct_tokenize(text))
    print()

vocab = Counter(tok for text in corpus for tok in word_punct_tokenize(text.lower()))
print("Vocabulary size for this tiny 6-sentence corpus:", len(vocab))
print("Words that share a root but get NO shared representation:")
print("  ", [w for w in vocab if "believ" in w])


### Code walkthrough — measuring the word-level tokenizer's failure modes

```python
def whitespace_tokenize(text):
    return text.split()
```

- **`text.split()`** with no arguments splits on any run of whitespace and discards empty strings. This
  is the crudest possible tokenizer: `"3:45pm."` stays glued together as one "word".

```python
def word_punct_tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text)
```

- **`re.findall(pattern, text)`** returns every non-overlapping match of `pattern` in `text`, in order.
- **`\w+`**: one or more "word" characters (letters, digits, underscore) — greedily grabs a whole
  alphanumeric run.
- **`|`**: alternation — try the left branch first, fall back to the right branch.
- **`[^\w\s]`**: a single character that is neither a word character nor whitespace, i.e. one piece of
  punctuation. This is why `"pipeline."` becomes `["pipeline", "."]` instead of one glued token.

```python
vocab = Counter(tok for text in corpus for tok in word_punct_tokenize(text.lower()))
```

- **`Counter(...)`** over a generator expression counts how many times each token string appears.
- **`text.lower()`** is applied before counting so that `"The"` and `"the"` map to the same vocabulary
  entry — a normalization choice every tokenizer has to make explicitly.

**What the output demonstrates:**

- `"Tokenization"` and `"Tokenisation"` become two *unrelated* vocabulary entries despite meaning the
  same thing — a word-level vocabulary cannot generalize across spelling variants.
- `"unbelievable"`, `"unbelievably"`, `"believable"`, `"believer"`, and `"disbelief"` all share the root
  `believ`, but a word-level tokenizer stores them as five independent, unrelated ids. The model has to
  re-learn "believe-related meaning" five times instead of once.
- A vocabulary built from just six sentences already needs dozens of entries; a vocabulary large enough
  to cover a real language (English alone has hundreds of thousands of word forms once you count
  inflections, typos, names, and numbers) would need to be enormous, and would *still* hit unseen words
  at inference time — the **out-of-vocabulary (OOV)** problem. Subword tokenization exists specifically
  to avoid both of these failures at once.


### Predict before you run

Below we tokenize `"internationalization"` character-by-character. Before running the cell: how many
characters long is the resulting sequence, and why is that a problem for a model with a fixed
context-window budget measured in tokens?


In [ ]:
word = "internationalization"
char_tokens = list(word)
print(char_tokens)
print("word-level length:", 1, "tokens   |   character-level length:", len(char_tokens), "tokens")


### Code walkthrough — the other extreme: character-level tokenization

`list("internationalization")` splits a Python string into its individual characters (a string is
already an iterable of characters, so `list()` just materializes that iterable).

Character-level tokenization has a tiny, fixed vocabulary (roughly the size of the alphabet plus
punctuation and digits) and therefore **never** has an out-of-vocabulary problem. Its cost is sequence
length: a 20-character word becomes 20 tokens instead of 1–3. Since a transformer's self-attention cost
scales quadratically with sequence length, and every model has a fixed maximum context length measured
in tokens, character-level tokenization wastes both compute and context budget on long inputs. Subword
tokenization is the compromise that keeps the vocabulary moderate *and* keeps common words to roughly one
token each, only splitting rare or morphologically complex words into a handful of pieces.


## 3. Meet a real tokenizer: `AutoTokenizer`

`AutoTokenizer.from_pretrained(...)` downloads and loads the exact tokenizer that shipped with a given
model checkpoint — vocabulary file, merge rules, special tokens, and all. Loading a tokenizer never
downloads model weights, so this section stays fast even without a GPU.


In [ ]:
from transformers import AutoTokenizer

bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "Tokenization isn't as simple as splitting on spaces."

tokens = bert_tok.tokenize(text)
ids = bert_tok.convert_tokens_to_ids(tokens)
decoded = bert_tok.decode(ids)

print("tokens :", tokens)
print("ids    :", ids)
print("decoded:", decoded)


### Code walkthrough — `tokenize` / `convert_tokens_to_ids` / `decode`

```python
bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
```

- **`from_pretrained("bert-base-uncased")`**: fetches the tokenizer files (`vocab.txt`,
  `tokenizer_config.json`, ...) from that repository on the Hub and reconstructs the exact tokenizer BERT
  was trained with. Using any other tokenizer with BERT's weights would map the same word to different
  ids than the ones the model learned embeddings for.

```python
tokens = bert_tok.tokenize(text)
```

- **`.tokenize(text)`** runs the tokenizer's algorithm (pre-tokenization + subword splitting) and returns
  a list of **token strings**, not ids yet. Notice `"isn't"` splits into pieces, and any subword that
  continues a previous one is prefixed with `##` (BERT's WordPiece convention meaning "no space before
  this piece"). `"Tokenization"` typically splits into `["token", "##ization"]` — a whole, common word
  (`token`) plus a common suffix (`##ization`) it has seen thousands of times across other words.

```python
ids = bert_tok.convert_tokens_to_ids(tokens)
```

- Looks each token string up in the vocabulary table and returns its integer index. This is a pure
  dictionary lookup — it does not re-run the splitting algorithm.

```python
decoded = bert_tok.decode(ids)
```

- **`.decode(ids)`** reverses the whole pipeline: ids → token strings → a single string, correctly
  re-joining `##`-prefixed continuations without inserting a space. Round-tripping through
  tokenize → convert → decode is a good sanity check when debugging tokenizer behavior.


In [ ]:
encoded = bert_tok(text)
print(encoded)
print()
print("input_ids     :", encoded["input_ids"])
print("as tokens     :", bert_tok.convert_ids_to_tokens(encoded["input_ids"]))
print("attention_mask:", encoded["attention_mask"])
print("token_type_ids:", encoded["token_type_ids"])


### Code walkthrough — calling the tokenizer directly: `tokenizer(text)`

`bert_tok(text)` is the interface you will use in practice — it does tokenize + convert + assemble in one
call and returns a `BatchEncoding` (a dict-like object) with three keys:

- **`input_ids`**: the integer ids, but now with two extra ids injected — one at the start, one at the
  end, that were **not** in the plain `.tokenize()` output.
- **`attention_mask`**: a list of `1`s the same length as `input_ids`, marking every position as "real
  content, attend to it" (as opposed to padding — see Section 6, where some positions become `0`).
- **`token_type_ids`**: a list of `0`s marking "this token belongs to the first (and here, only)
  segment". BERT was pretrained on **pairs** of segments (e.g. question + context) separated by a
  special token, with `1`s marking the second segment; for single-sentence input everything is segment
  `0`.

The two extra ids at the start and end decode to `[CLS]` and `[SEP]` — BERT's **special tokens**:

- **`[CLS]`** ("classification"): prepended to every input; BERT's pretraining uses this position's final
  hidden state as a summary of the whole sequence for sentence-level tasks.
- **`[SEP]`** ("separator"): marks the end of a segment, and separates two segments when a pair is given.

`AutoTokenizer` inserts these automatically because `add_special_tokens=True` is the default — this is
also why `bert_tok(text)["input_ids"]` is two elements longer than `convert_tokens_to_ids(tokenize(text))`
from the previous cell.


### Exercise 3.1

Load `AutoTokenizer.from_pretrained("gpt2")` and tokenize the same `text` variable from
above. GPT-2 uses a decoder-only, causal-LM design and was never trained on the two-segment task BERT
was. Before running anything: do you expect `token_type_ids` to appear in GPT-2's output? What about
`[CLS]`/`[SEP]`-equivalent special tokens? Then check your prediction with code.


## 4. Subword algorithms from the ground up

`bert_tok.tokenize(...)` felt like magic in the previous section. This section removes the magic by
implementing the core idea — **Byte-Pair Encoding (BPE)** — in about 20 lines of plain Python, on a
corpus small enough to trace by hand.

**The BPE training algorithm, in words:**

1. Start with every word split into individual characters (plus an end-of-word marker so the model can
   tell `"est"` at the end of `"newest"` apart from `"est"` at the start of a word).
2. Count every adjacent **pair** of symbols across the whole corpus.
3. Merge the single most frequent pair into one new symbol, and record that merge rule.
4. Repeat steps 2–3 for a fixed number of merges (this merge count is the main knob controlling final
   vocabulary size).

The learned list of merge rules, applied greedily in the order they were learned, *is* the tokenizer.


In [ ]:
from collections import Counter, defaultdict

# A tiny corpus, deliberately repetitive so merges are easy to trace by hand.
# Each word is represented as a tuple of characters plus an end-of-word marker "</w>".
word_freqs = Counter({
    "low": 5,
    "lower": 2,
    "lowest": 2,
    "newer": 6,
    "wider": 3,
    "new": 2,
})

corpus_symbols = {
    tuple(word) + ("</w>",): freq for word, freq in word_freqs.items()
}
print("Initial character-level representation:")
for word, freq in corpus_symbols.items():
    print(f"  {word}  (freq={freq})")


### Code walkthrough — representing the corpus for BPE training

```python
word_freqs = Counter({"low": 5, "lower": 2, ...})
```

- A word-frequency table stands in for a real corpus. Real BPE trainers (including the one in Section 5)
  build this same table from millions of words; using six here keeps every merge traceable by hand.

```python
corpus_symbols = {tuple(word) + ("</w>",): freq for word, freq in word_freqs.items()}
```

- **`tuple(word)`**: splits the string into a tuple of single characters, e.g. `"low"` → `("l","o","w")`.
- **`+ ("</w>",)`**: appends an explicit end-of-word marker. Without it, the substring `"er"` inside
  `"newer"` (mid-word) and the suffix `"er"` (end of word, as in a hypothetical `"er"` standalone) would
  be indistinguishable to the merge-counting step; the marker lets BPE learn that *word-final* `"er"` is
  a common, meaningful suffix while keeping that fact separate from `"er"` appearing elsewhere.
- The dictionary keys are these symbol-tuples; the values are how many times that word occurs in the
  corpus, carried over unchanged from `word_freqs`.


In [ ]:
def get_pair_counts(corpus_symbols):
    pair_counts = Counter()
    for symbols, freq in corpus_symbols.items():
        for a, b in zip(symbols, symbols[1:]):
            pair_counts[(a, b)] += freq
    return pair_counts

def merge_pair(pair, corpus_symbols):
    a, b = pair
    merged = a + b
    new_corpus = {}
    for symbols, freq in corpus_symbols.items():
        new_symbols = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                new_symbols.append(merged)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        new_corpus[tuple(new_symbols)] = new_corpus.get(tuple(new_symbols), 0) + freq
    return new_corpus

num_merges = 8
merge_rules = []
current_corpus = dict(corpus_symbols)

for step in range(num_merges):
    pair_counts = get_pair_counts(current_corpus)
    if not pair_counts:
        break
    best_pair = max(pair_counts, key=pair_counts.get)
    current_corpus = merge_pair(best_pair, current_corpus)
    merge_rules.append(best_pair)
    print(f"merge {step + 1}: {best_pair}  (count={pair_counts[best_pair]})  ->  {best_pair[0] + best_pair[1]!r}")

print()
print("Final symbol sequences:")
for symbols, freq in current_corpus.items():
    print(f"  {symbols}  (freq={freq})")


### Code walkthrough — the merge loop, argument by argument

```python
def get_pair_counts(corpus_symbols):
    ...
    for a, b in zip(symbols, symbols[1:]):
        pair_counts[(a, b)] += freq
```

- **`zip(symbols, symbols[1:])`**: a standard idiom for iterating over every *adjacent* pair in a
  sequence — `symbols[1:]` is the same tuple shifted left by one, so `zip` produces
  `(symbols[0], symbols[1]), (symbols[1], symbols[2]), ...`.
- Each pair's count is incremented by `freq`, not by `1` — a pair occurring once inside a word that
  appears 6 times in the corpus counts as 6 occurrences, matching how often the model will actually see
  it.

```python
def merge_pair(pair, corpus_symbols):
    ...
    while i < len(symbols):
        if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
            new_symbols.append(merged)
            i += 2
        else:
            new_symbols.append(symbols[i])
            i += 1
```

- A manual index-based scan (rather than a regex or `str.replace`) because symbols are tuple elements,
  not characters in a string, and because merges must not overlap: after consuming a matched pair we
  jump `i` forward by 2 (`i += 2`) so the same characters can't be re-merged inside a single pass.

```python
best_pair = max(pair_counts, key=pair_counts.get)
```

- **`max(iterable, key=...)`** finds the element of `pair_counts` (its keys, i.e. the pairs) that
  maximizes `key(pair)`. **`key=pair_counts.get`** uses the dictionary's own `.get` method as the scoring
  function, so this line reads as "the pair with the highest count" — this is the one BPE rule at the
  heart of the whole algorithm: *always merge the currently most frequent adjacent pair.*

**Reading the trace:** the first merge is almost always `('e', 'r')` — it appears in `newer` (freq 6),
`wider` (freq 3), and `lower` (freq 2), for a combined count higher than any other pair. A few merges
later you should see `('er', '</w>')` fire, producing the whole-suffix symbol `er</w>` — the algorithm has
just discovered, purely from frequency statistics and with no linguistic knowledge, that "words ending in
er" is a recurring pattern worth its own symbol. This is exactly the mechanism (scaled to a corpus of
billions of words and tens of thousands of merges) behind every production BPE tokenizer, including
GPT-2's.


### Predict before you run

`merge_rules` now holds the learned merges **in the order they were learned**. To tokenize a brand-new
word, you apply the same merges to its characters, in that same order — never re-deriving frequencies.
Before running the next cell: will `"lowest"` end up as more or fewer tokens than `"new"`? Both appeared
in the training corpus, but with different frequencies and different neighboring letters.


In [ ]:
def apply_bpe(word, merge_rules):
    symbols = list(word) + ["</w>"]
    for a, b in merge_rules:
        i = 0
        new_symbols = []
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                new_symbols.append(a + b)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        symbols = new_symbols
    return symbols

for w in ["lowest", "new", "widest"]:  # "widest" never appeared in training at all
    print(f"{w!r:>10} -> {apply_bpe(w, merge_rules)}")


### Code walkthrough — applying learned merges to new words

`apply_bpe` is almost identical to `merge_pair` from the training loop, but now it walks the *fixed*
`merge_rules` list **in order** instead of recomputing the globally best pair each time. This ordering
matters: a merge learned early (because it was frequent early on) takes priority over one learned late,
even if applying them in a different order would sometimes produce a different split.

`"widest"` was never seen during training, yet `apply_bpe` still produces a sensible split — it reuses
whatever merges *do* apply (e.g. an `'e','s'` or `'s','t'` merge learned from `"lowest"`/`"newer"`-family
words) and leaves the rest as single characters. **This is BPE's core guarantee: every input can be
tokenized, because the algorithm can always fall back to individual characters.** There is no
out-of-vocabulary problem, only "more splitting than usual" for unfamiliar text — a graceful degradation
that word-level tokenizers cannot offer.


### Exercise 4.1

Increase `num_merges` in the training cell above from 8 to 20 and re-run both the
training cell and the `apply_bpe` cell. What happens to `"lowest"`'s token count as more merges are
learned? At what point (roughly) does further increasing `num_merges` stop shortening common words and
start only adding merges for words already close to whole tokens? This is the same trade-off production
tokenizers face when choosing a target vocabulary size (typically 32k–128k for LLMs): more merges means
shorter sequences but a larger embedding table and slower convergence on rare pieces.


## 5. BPE vs. WordPiece vs. Unigram: same idea, different scoring rule

All three families you'll meet in practice merge or split *subwords* rather than characters or whole
words; they differ in **how they decide which merge/split to make**:

| Algorithm | Merge/split criterion | Used by |
|---|---|---|
| **BPE** | Merge the most *frequent* adjacent pair (exactly what you just implemented) | GPT-2, GPT-4 (byte-level variant), RoBERTa |
| **WordPiece** | Merge the pair that most increases the *likelihood* of the training data under a unigram language model — frequency divided by the frequencies of its parts, roughly preferring pairs whose parts are individually rare but which co-occur a lot | BERT, DistilBERT, Electra |
| **Unigram (SentencePiece)** | Start from a *large* candidate vocabulary and iteratively *remove* the pieces that hurt a probabilistic model of the corpus least, until reaching the target size — the reverse direction from BPE/WordPiece | T5, ALBERT, XLNet, Llama, Gemma |

Let's tokenize the same sentence with one tokenizer from each family and compare the actual output.


In [ ]:
from transformers import AutoTokenizer

sentence = "The unbelievably fast transformer tokenizes internationalization."

tokenizer_families = {
    "gpt2 (byte-level BPE)": "gpt2",
    "bert-base-uncased (WordPiece)": "bert-base-uncased",
    "xlnet-base-cased (Unigram/SentencePiece)": "xlnet-base-cased",
}

for label, checkpoint in tokenizer_families.items():
    tok = AutoTokenizer.from_pretrained(checkpoint)
    pieces = tok.tokenize(sentence)
    print(f"{label}")
    print(f"  {pieces}")
    print(f"  -> {len(pieces)} tokens")
    print()


### Code walkthrough — reading the three outputs

Loading three tokenizers and calling `.tokenize(sentence)` on each reuses exactly the interface from
Section 3 — the only thing that changes is the `checkpoint` string passed to `from_pretrained`.

What to look for in the printed pieces:

- **GPT-2** tokens often carry a visible leading-space marker (rendered as `Ġ`, a byte-level stand-in for
  a space — more on this in Section 6) attached to the *front* of a piece, e.g. `Ġtransformer`, because
  GPT-2 was trained on raw web text where whitespace is meaningful and must be preserved exactly through
  tokenization and decoding.
- **BERT's WordPiece** pieces after the first one in a word are prefixed with `##`, e.g.
  `["token", "##izes"]`, marking "glue me directly onto the previous piece, no space."
- **XLNet's Unigram/SentencePiece** pieces use a leading `▁` (U+2581, a dedicated "space" symbol, distinct
  from underscore) to mark the *start* of a new word, which is the opposite convention from WordPiece's
  `##` — SentencePiece marks word starts, WordPiece marks word continuations.

None of these conventions is "more correct" than another; they are different choices for the same
underlying problem (how to make tokenization losslessly reversible, i.e. `decode(tokenize(x)) == x`,
including whitespace). What matters practically is that **you never mix a decode convention from one
tokenizer with ids from another** — always load the tokenizer that shipped with the model.


## 6. Byte-level BPE: how GPT-2/GPT-4 avoid unknown tokens entirely

The character-level BPE you implemented in Section 4 can still hit an unknown symbol at inference time —
if it merges over Unicode *characters* and someone sends an emoji or a rare script your training corpus
never contained, the algorithm has no rule for it.

**Byte-level BPE** sidesteps this completely with one change: run the *exact same* merge algorithm over
the raw **UTF-8 bytes** of the text instead of over Unicode characters. There are only 256 possible byte
values, so the base vocabulary (before any merges) is fixed at 256 symbols and is guaranteed to cover
*any* text in *any* language or script, including emoji — because everything is bytes underneath. This is
the trick GPT-2, GPT-3/4, and most modern open LLMs (Llama, Qwen, Mistral) use.


In [ ]:
from transformers import AutoTokenizer

gpt2_tok = AutoTokenizer.from_pretrained("gpt2")

samples = [
    "Hello, world!",
    "Tokenization 🤯 works on emoji too.",
    "日本語のテキストも処理できます.",  # Japanese: "Japanese text can also be processed."
]

for text in samples:
    tokens = gpt2_tok.tokenize(text)
    ids = gpt2_tok.encode(text)
    print(f"{text!r}")
    print(f"  tokens ({len(tokens)}): {tokens}")
    print(f"  ids: {ids}")
    print(f"  round-trip decode: {gpt2_tok.decode(ids)!r}")
    print()


### Code walkthrough — byte-level tokens and the `Ġ` / weird-character convention

```python
ids = gpt2_tok.encode(text)
```

- **`.encode(text)`** is shorthand for tokenize + convert_tokens_to_ids in one call, returning a plain
  list of ints (unlike `tokenizer(text)`, it skips building the full `BatchEncoding` with an attention
  mask — useful when you only need ids, e.g. for counting tokens).

Look closely at the printed `tokens` list for the emoji and Japanese examples: you'll see pieces that
don't look like readable text at all — sequences like `å` or `ĥ`-style glyphs. **These are not
Unicode characters being tokenized**; they are individual UTF-8 **bytes**, each remapped to a printable
Unicode character purely so the merge algorithm and any downstream code can treat every byte value as an
ordinary, visible symbol (raw bytes 0–31 and 127–160-ish aren't safely printable/whitespace-safe as-is).
The 🤯 emoji, for instance, is 4 bytes in UTF-8, and if it never appeared often enough in GPT-2's training
data to earn its own merge, it will show up as up to 4 separate byte-tokens — noticeably less efficient
than a common English word, but crucially **never an error**.

The **`Ġ`** you'll see prefixing many tokens is this same byte-remapping trick applied to the space
character (byte value 0x20): GPT-2's pre-tokenizer attaches the preceding space to the *start* of the
next word before BPE runs, so `" world"` (with its leading space) becomes its own trainable unit, distinct
from word-initial `"world"` with no space. This is why GPT-2 tokenization is **sensitive to leading
whitespace** — see the pitfall in Section 8.

The round-trip `decode(encode(text)) == text` holding exactly, even for the emoji and Japanese text, is
the payoff: byte-level BPE is lossless by construction, because every possible byte sequence is
representable, unlike character-level vocabularies which must either drop or replace unfamiliar
characters with an `[UNK]` token.


In [ ]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4")
text = "Tokenization 🤯 works on emoji too."
ids = enc.encode(text)
print("gpt-4 (tiktoken) ids   :", ids)
print("gpt-4 (tiktoken) tokens:", [enc.decode([i]) for i in ids])
print("gpt-4 (tiktoken) count :", len(ids))


### Code walkthrough — `tiktoken`, the tokenizer behind OpenAI's API

`tiktoken` is a standalone, dependency-light library implementing the exact same byte-level BPE family,
used to compute token counts for OpenAI models without needing the `transformers` library or a network
call to a model repo.

- **`tiktoken.encoding_for_model("gpt-4")`**: looks up which named byte-level BPE encoding (e.g.
  `cl100k_base`) a given model uses, and loads its merge rules and byte-vocabulary.
- **`enc.encode(text)`**: returns integer ids directly (no separate `.tokenize()` step is exposed in this
  library's API — encoding straight to ids is `tiktoken`'s primary use case, since its main job is
  **counting/estimating tokens for billing and context-length budgeting**, not linguistic analysis).
- **`enc.decode([i])`** called once per id, inside the list comprehension, recovers each individual
  token's printable text — useful for inspection, even though `tiktoken` does not expose a
  `convert_ids_to_tokens`-style method the way `transformers` tokenizers do.

Practically: **token count, not character or word count, is what LLM APIs bill by and what context
windows are measured in.** `len(enc.encode(text))` is the standard way to estimate the cost of a prompt
before sending it.


### Exercise 6.1

Pick a sentence in a non-Latin script you know (Persian, Arabic, Chinese, etc.), or
reuse the Japanese example above. Compare `len(gpt2_tok.encode(sentence))` against
`len(gpt2_tok.encode(english_translation))` for a same-meaning English sentence of similar length. Which
one uses more tokens per word of meaning, and why does that matter for anyone paying per-token for API
access, or for a model's effective context window, in a non-English deployment?


## 7. Training your own tokenizer with the `tokenizers` library

Section 4 trained toy BPE by hand on six words. The `tokenizers` library runs the same algorithm,
production-grade and Rust-fast, on a real corpus of any size. We'll train a small BPE tokenizer on a
Hugging Face dataset and then load it back through `AutoTokenizer`/`PreTrainedTokenizerFast` so it's
usable exactly like any pretrained tokenizer from the Hub.


In [ ]:
from datasets import load_dataset

raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train[:2000]")
raw = raw.filter(lambda ex: len(ex["text"].strip()) > 0)
print(raw)
print(raw[10])


### Code walkthrough — a small real training corpus

- **`load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train[:2000]")`**: reuses the
  slicing pattern from Tutorial 0 — `wikitext-2-raw-v1` is a modest Wikipedia-derived text dataset (hosted
  under the `Salesforce` namespace on the Hub), and `train[:2000]` takes only its first 2,000 rows, enough
  to see real merge behavior without a long training time.
- **`.filter(lambda ex: len(ex["text"].strip()) > 0)`**: WikiText ships with many blank-line rows used as
  paragraph separators in the original format; `.filter` keeps only rows whose predicate returns `True`,
  here dropping rows that are empty once leading/trailing whitespace is stripped.


In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=8000,
    min_frequency=2,
    special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]"],
    show_progress=True,
)

def batch_iterator(dataset, batch_size=1000):
    for i in range(0, len(dataset), batch_size):
        yield dataset[i : i + batch_size]["text"]

tokenizer.train_from_iterator(batch_iterator(raw), trainer=trainer)
print("Trained vocabulary size:", tokenizer.get_vocab_size())


### Code walkthrough — `Tokenizer`, `BpeTrainer`, and `train_from_iterator`, argument by argument

```python
tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
```

- **`models.BPE(...)`**: selects the BPE algorithm as the tokenizer's core model (the library also offers
  `models.WordPiece` and `models.Unigram` — swapping this one line changes the entire algorithm family
  while keeping the rest of the pipeline identical).
- **`unk_token="[UNK]"`**: the fallback token used only if a byte/character genuinely cannot be
  represented — with byte-level pre-tokenization (next line) this essentially never fires, since every
  byte value is representable, but the model still requires a designated symbol for the theoretical case.

```python
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
```

- The **pre-tokenizer** runs *before* BPE merging and decides the initial split points (e.g. "split on
  whitespace boundaries, and remap every byte to a printable surrogate character" — this is the
  byte-level scheme from Section 6, now supplied as a configurable component instead of hidden inside a
  pretrained checkpoint).
- **`add_prefix_space=False`**: controls whether a space is silently added before the input if it doesn't
  already start with one, which affects whether the very first word of a text is treated as
  "word-initial" (with a leading-space marker) the same way later words are. GPT-2 itself uses
  `add_prefix_space=False` for single texts; setting it `True` is common when tokenizing a list of
  already-split words so the first one isn't treated differently from the rest.

```python
tokenizer.decoder = decoders.ByteLevel()
```

- The **decoder** must match the pre-tokenizer's encoding scheme so that ids → text reverses the byte
  remapping correctly; forgetting to set a matching decoder is a common bug that produces garbled output
  even when encoding itself works fine.

```python
trainer = trainers.BpeTrainer(
    vocab_size=8000,
    min_frequency=2,
    special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]"],
    show_progress=True,
)
```

- **`vocab_size=8000`**: the target vocabulary size — training stops once this many symbols (base bytes +
  merges) have been produced. This is the single most important hyperparameter of tokenizer training; it
  is chosen once per model family and never changed afterward, since changing it changes the meaning of
  every id.
- **`min_frequency=2`**: a candidate merge must occur at least this many times in the corpus to be
  accepted, preventing the vocabulary from being spent on one-off noise (typos, corrupted text).
- **`special_tokens=[...]`**: reserved symbols guaranteed a slot in the vocabulary regardless of their
  frequency in the corpus (here: unknown, padding, beginning/end-of-sequence — the roles you'll assign to
  them explicitly in the next cell).
- **`show_progress=True`**: prints a progress bar during training — purely cosmetic, useful here only
  because training even a small BPE model takes a few seconds and it's reassuring to see it moving.

```python
def batch_iterator(dataset, batch_size=1000):
    for i in range(0, len(dataset), batch_size):
        yield dataset[i : i + batch_size]["text"]

tokenizer.train_from_iterator(batch_iterator(raw), trainer=trainer)
```

- **`train_from_iterator`** expects an iterable of strings (or batches of strings) rather than the whole
  corpus materialized as one object — `batch_iterator` yields 1,000-row slices of text at a time, which
  scales to corpora far too large to hold as a single Python list.
- **`trainer=trainer`**: passes the hyperparameter object from above; the model architecture
  (`models.BPE`) and the training hyperparameters (`BpeTrainer`) are deliberately two separate objects,
  so the same trainer class works no matter which of BPE/WordPiece/Unigram you picked at construction.


In [ ]:
sample = "The tokenizer's vocabulary determines what unbelievably efficient models can learn."
out = tokenizer.encode(sample)
print("tokens:", out.tokens)
print("ids   :", out.ids)
print("decode:", tokenizer.decode(out.ids))


### Code walkthrough — using the freshly trained tokenizer

- **`tokenizer.encode(sample)`** on a raw `tokenizers.Tokenizer` (as opposed to a `transformers`
  `AutoTokenizer`) returns an `Encoding` object with `.tokens`, `.ids`, `.attention_mask`, etc. as
  attributes rather than dictionary keys — a small API difference worth knowing when you drop down to
  this lower-level library.
- Compare this tokenizer's split of `"unbelievably"` against GPT-2's split of the same word back in
  Section 6 — because both are byte-level BPE, the *mechanism* is identical, but the *specific merges*
  differ because this one was trained on 2,000 rows of WikiText instead of GPT-2's much larger web-scale
  corpus. Vocabulary content is a direct fingerprint of training data.


In [ ]:
from transformers import PreTrainedTokenizerFast

fast_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    unk_token="[UNK]",
    pad_token="[PAD]",
    bos_token="[BOS]",
    eos_token="[EOS]",
)

batch = fast_tokenizer(
    ["Short text.", "A somewhat longer sentence to force padding to kick in."],
    padding=True,
    return_tensors="pt",
)
print(batch["input_ids"])
print(batch["attention_mask"])


### Code walkthrough — wrapping a trained `Tokenizer` as a Hugging Face tokenizer

```python
fast_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    unk_token="[UNK]",
    pad_token="[PAD]",
    bos_token="[BOS]",
    eos_token="[EOS]",
)
```

- **`PreTrainedTokenizerFast`**: the same base class every fast `AutoTokenizer` in `transformers` actually
  instantiates behind the scenes — this line makes that explicit by constructing it directly from our
  own `tokenizer_object` instead of from a Hub repository.
- **`tokenizer_object=tokenizer`**: the `tokenizers.Tokenizer` we trained above, plugged in as the
  underlying engine.
- **`unk_token=`, `pad_token=`, `bos_token=`, `eos_token=`**: these tell `transformers` which of the
  special-token *strings* play which *role*. This is a separate step from `special_tokens=[...]` in the
  trainer — that reserved the vocabulary slots; this assigns semantic roles to them so that
  `fast_tokenizer.pad_token_id`, padding logic, and generation code know which id to use for which
  purpose.

The final call reuses the padding/`return_tensors` interface previewed here and covered in full in
Section 8 next — a trained-from-scratch tokenizer supports exactly the same `__call__` interface as any
pretrained one, because they share the same base class. This is also how you would save
(`fast_tokenizer.save_pretrained("my-tokenizer")`) and later reload
(`AutoTokenizer.from_pretrained("my-tokenizer")`) a tokenizer you trained yourself for a real project,
e.g. before pretraining a model on a new language or domain.


## 8. The practical interface: `padding`, `truncation`, `max_length`, `return_tensors`

Real training and inference code always tokenizes **batches**, not single strings, and batches need
uniform-length tensors. This section reads every argument that controls that shape.


In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("bert-base-uncased")

batch_texts = [
    "Short.",
    "A medium length sentence for the batch.",
    "This is a considerably longer sentence, included specifically to force truncation once we set a small max_length.",
]

encoded = tok(
    batch_texts,
    padding=True,
    truncation=True,
    max_length=12,
    return_tensors="pt",
)

print("input_ids shape:", encoded["input_ids"].shape)
for row in encoded["input_ids"]:
    print(" ", row.tolist(), "->", tok.convert_ids_to_tokens(row.tolist()))
print()
print("attention_mask:")
for row in encoded["attention_mask"]:
    print(" ", row.tolist())


### Code walkthrough — `tokenizer(batch, padding=, truncation=, max_length=, return_tensors=)`

```python
encoded = tok(
    batch_texts,
    padding=True,
    truncation=True,
    max_length=12,
    return_tensors="pt",
)
```

- **first positional argument: `batch_texts`** — a **list** of strings instead of one string switches the
  tokenizer into batch mode automatically; every returned field becomes a list-of-lists (or a 2D tensor
  with `return_tensors` set), one row per input string.
- **`padding=True`**: pads every sequence in the batch to the length of the **longest** sequence in that
  batch (equivalent to `padding="longest"`), by appending `pad_token_id` (BERT: `[PAD]`, id `0`) until
  all rows match. Padding exists because tensors must be rectangular, but padded positions carry no real
  content.
- **`truncation=True`**: cuts any sequence longer than `max_length` down to exactly `max_length` tokens
  (by default from the end, dropping the tail) *before* padding is applied — this is why the third,
  deliberately long sentence above is cut short.
- **`max_length=12`**: the ceiling both truncation and (when `padding="max_length"` instead of `True`)
  padding target. `12` is intentionally tiny here purely to make truncation visible in a short printed
  output; a real model's `max_length` matches its trained context window (e.g. 512 for base BERT, tens of
  thousands for modern LLMs).
- **`return_tensors="pt"`**: returns PyTorch tensors instead of plain Python lists. Common alternatives:
  `"tf"` for TensorFlow tensors, `"np"` for NumPy arrays, or omitting the argument entirely to get plain
  Python lists (the default, and often the right choice for inspection/debugging, as in every earlier
  section of this notebook).

**Reading the printed output:** the padded positions all show `id 0` and decode to `[PAD]`, and the
corresponding `attention_mask` entries are `0` at exactly those positions — telling the model's attention
mechanism to ignore them entirely, so a batch's padding never influences the (real) predictions for
shorter sequences sharing that batch.


### Predict before you run

If we set `padding="max_length"` instead of `padding=True`, with the same `max_length=12`, what will
change about the shape of `encoded["input_ids"]` compared to the cell above — specifically, will the
*shortest* sentence ("Short.") be padded to the same length as before, a shorter length, or a longer one?


In [ ]:
encoded_fixed = tok(
    batch_texts,
    padding="max_length",
    truncation=True,
    max_length=12,
    return_tensors="pt",
)
print("input_ids shape:", encoded_fixed["input_ids"].shape)
print(encoded_fixed["input_ids"][0].tolist(), "  (row for the shortest sentence)")


### Code walkthrough — `padding="longest"` vs. `padding="max_length"`

- **`padding=True`** (equivalently `padding="longest"`) pads to whatever the **longest sequence in this
  particular batch** happens to be — efficient, but the padded length can vary from batch to batch.
- **`padding="max_length"`** always pads to the fixed value you pass as `max_length`, regardless of how
  long the actual sequences in the batch are — every batch produces the exact same tensor shape, which
  some deployment settings (e.g. certain compiled/exported models, or fixed-shape hardware like some TPU
  configurations) require, at the cost of more wasted computation on padding when most inputs are much
  shorter than `max_length`.

The shortest sentence ("Short.") is padded to length 12 either way in this example, because `12` is also
the value we chose for `max_length` and no sentence in the batch is shorter than that after tokenization
— but with a smaller batch-max (say if every sentence were 4 tokens long), `padding=True` would stop
padding at 4 while `padding="max_length"` would still pad all the way to 12.


## 9. Special tokens for chat models: `apply_chat_template`

Instruction-tuned / chat LLMs (Qwen, Llama-Instruct, ChatGPT-style models) are trained on conversations
wrapped in a specific format of role markers and special tokens — sending them plain, unformatted text
is a very common source of poor-quality output, because the model has essentially never seen unformatted
input during training. `apply_chat_template` builds the correct format for you from a list of role/content
dictionaries, without you needing to memorize each model family's exact markup.


In [ ]:
from transformers import AutoTokenizer

chat_tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")

messages = [
    {"role": "system", "content": "You are a concise assistant for an NLP course."},
    {"role": "user", "content": "In one sentence, what is byte-level BPE?"},
]

prompt_string = chat_tok.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
print(prompt_string)


### Code walkthrough — `apply_chat_template(messages, tokenize=, add_generation_prompt=)`

```python
messages = [
    {"role": "system", "content": "..."},
    {"role": "user", "content": "..."},
]
```

- A conversation is a **list of dicts**, each with a **`role`** (`"system"`, `"user"`, or `"assistant"`)
  and **`content`** (the message text). This structure is shared across essentially every chat-tuned
  model in `transformers`, even though the underlying markup each model turns it into differs.

```python
prompt_string = chat_tok.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
```

- **`tokenize=False`**: return the formatted **string**, not token ids — useful here purely so we can
  print and inspect the exact markup being generated. In real inference code you would normally pass
  `tokenize=True` (or omit it, since it's commonly the default) and get ids directly, ready to feed to
  `model.generate(...)`.
- **`add_generation_prompt=True`**: appends the special marker sequence that tells the model "now it's
  your turn to speak as the assistant" (e.g. an assistant-role opening tag with no content yet). Without
  this, the formatted prompt would end right after the user's turn, and the model would be more likely to
  continue writing a *user* turn instead of answering — a subtle but common bug when integrating chat
  models manually.

Every model family defines its own **chat template** (a Jinja2 template stored in the tokenizer's
config) that decides the exact special tokens and role markers produced — this is precisely why loading
the *matching* tokenizer for a chat model, not a generic one, matters even more than usual: the template
itself is part of what the model was fine-tuned to expect.


## 10. Common tokenization pitfalls in real applications

These four issues account for a large share of "the model gives worse answers than it should" bug
reports in practice.


In [ ]:
gpt2_tok = AutoTokenizer.from_pretrained("gpt2")

print("with leading space   :", gpt2_tok.tokenize(" world"))
print("without leading space:", gpt2_tok.tokenize("world"))
print()
print("digits, glued  :", gpt2_tok.tokenize("3.14159"))
print("digits, spaced :", gpt2_tok.tokenize("3 . 1 4 1 5 9"))


### Code walkthrough — pitfall 1: leading-space sensitivity; pitfall 2: digit splitting

**Leading-space sensitivity:** `" world"` and `"world"` are tokenized by GPT-2-family BPE into
**different token ids**, because (as covered in Section 6) the leading space is fused onto the *front* of
the following word before merges are applied. This bites people constructing prompts by string
concatenation — `prompt + " " + continuation` vs. `prompt + continuation` can produce meaningfully
different token sequences even though a human reader sees "the same text with a space added where one
was obviously needed."

**Digit splitting:** byte-level BPE tokenizers commonly split multi-digit numbers inconsistently — some
short common numbers get their own token, longer or less common ones get split digit-by-digit or in
uneven groups (`"3.14159"` splitting differently from `"3 . 1 4 1 5 9"` above is one illustration; try
`"12345"` vs `"54321"` and you'll often see different, arbitrary-looking splits too). This is a
significant, well-documented reason LLMs are historically unreliable at multi-digit arithmetic: the model
never sees numbers as a consistent positional (ones/tens/hundreds) representation — it sees whatever
chunks the tokenizer's training-frequency statistics happened to produce. Some newer tokenizers (e.g. in
Llama) deliberately force single-digit tokenization for exactly this reason.


In [ ]:
bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
gpt2_tok = AutoTokenizer.from_pretrained("gpt2")

text = "Tokenization mismatch example."
bert_ids = bert_tok.encode(text)
print("BERT ids           :", bert_ids)
print("Decoded with GPT-2 :", gpt2_tok.decode(bert_ids))


### Code walkthrough — pitfall 3: tokenizer/model mismatch

Feeding BERT's ids into GPT-2's `.decode()` "works" in the sense that it doesn't raise an error — every
integer in range is a valid lookup key into *some* vocabulary entry — but the output is meaningless,
because id `2000` (for example) refers to a completely different token string in each vocabulary. This is
the single most important rule in this notebook to internalize: **a tokenizer and a model must always be
loaded from the same checkpoint/repository.** `AutoTokenizer.from_pretrained("bert-base-uncased")` paired
with `AutoModel.from_pretrained("gpt2")` is a bug that silently produces nonsense rather than an error,
which makes it more dangerous than a bug that crashes loudly.


In [ ]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4")

pairs = [
    ("English", "The quick brown fox jumps over the lazy dog."),
    ("Persian", "روباه قهوه‌ای سریع از روی سگ تنبل می‌پرد."),
]

for lang, text in pairs:
    n = len(enc.encode(text))
    print(f"{lang:8s}: {n:3d} tokens for {len(text):3d} characters  ({n/len(text):.2f} tokens/char)")


### Code walkthrough — pitfall 4: multilingual token inefficiency

Most widely-used tokenizer vocabularies are trained on corpora dominated by English and other
high-resource, Latin-script languages, because that's what's most abundant on the web. As a direct
consequence, text in lower-resource languages or non-Latin scripts frequently falls back to much smaller
byte-level fragments — the "tokens/char" ratio printed above is typically noticeably higher for a
non-Latin-script sentence than for an English sentence of comparable meaning.

This has two concrete, practical consequences: (1) **cost** — API pricing is per-token, so the same
sentence can cost several times more to send in some languages than in English; and (2) **effective
context window** — if a model's context limit is, say, 8,000 tokens, a language that needs 3× more
tokens per sentence effectively gets roughly a third of the usable context length for the same amount of
actual content. This is an active, ongoing area of tokenizer-design research (e.g. training
larger/multilingual-balanced vocabularies specifically to close this gap).


## 11. Mini-lab — build a tokenizer efficiency report

Using only tools introduced in this notebook, produce a short comparison across **at least three
tokenizers** (e.g. `gpt2`, `bert-base-uncased`, `xlnet-base-cased`, or a chat tokenizer such as
`Qwen/Qwen3-0.6B`) and **at least two languages** (English plus one language you know, or reuse the
Persian example above):

1. For each (tokenizer, language) pair, tokenize the same 3–5 sentences of comparable meaning and length.
2. Report the average tokens-per-word (or tokens-per-character) ratio for each pair.
3. Identify which tokenizer is most efficient for which language, and connect that back to Section 10's
   discussion of cost and effective context window.
4. In two sentences: if you were choosing a tokenizer for a product deployed primarily in the
   *non-English* language you tested, what would you look for or do differently?

### Starter code


In [ ]:
from transformers import AutoTokenizer

report_tokenizers = {
    "gpt2": AutoTokenizer.from_pretrained("gpt2"),
    "bert-base-uncased": AutoTokenizer.from_pretrained("bert-base-uncased"),
    # add more checkpoints here
}

report_sentences = {
    "English": [
        "The weather is nice today.",
        "I am learning about tokenization.",
    ],
    # add another language's sentences here, matched in meaning and length
}

for lang, sentences in report_sentences.items():
    for name, tok in report_tokenizers.items():
        total_tokens = sum(len(tok.tokenize(s)) for s in sentences)
        total_words = sum(len(s.split()) for s in sentences)
        print(f"{lang:10s} | {name:20s} | {total_tokens/total_words:.2f} tokens/word")


## 12. Where each later tutorial fits

| Topic | Library | Notebook |
|---|---|---|
| Transformer internals: hidden states, attention, KV cache | `transformers` | Tutorial 3 |
| Fine-tuning a model on your own data | `transformers` + `datasets` + `Trainer` | later tutorial |
| Running on multiple GPUs / mixed precision | `accelerate` | later tutorial |
| Efficient inference: quantization, batching, FlashAttention | `transformers` + `accelerate` | later tutorial |

Everything from here on assumes you can read a tokenizer's output confidently: token counts, special
tokens, padding/attention masks, and chat-template formatting are all inputs the model internals in
Tutorial 3 build directly on top of.


## Glossary — quick reference

| Term | Meaning |
|---|---|
| **Token** | The smallest unit a model reads/writes — a whole word, a subword piece, or a byte, depending on the tokenizer. |
| **Vocabulary** | The fixed set of all token strings a tokenizer can produce, each mapped to an integer id. |
| **BPE (Byte-Pair Encoding)** | Subword algorithm that iteratively merges the most *frequent* adjacent symbol pair. |
| **WordPiece** | Subword algorithm (BERT) that merges the pair maximizing training-data likelihood, not raw frequency. |
| **Unigram / SentencePiece** | Subword algorithm (T5, Llama) that starts from a large vocabulary and *prunes* down to the target size. |
| **Byte-level BPE** | BPE run over raw UTF-8 bytes instead of characters — guarantees no out-of-vocabulary input. |
| **OOV (out-of-vocabulary)** | A word-level tokenizer's failure mode: an input word has no vocabulary entry at all. |
| **Special tokens** | Reserved symbols with a role beyond plain text, e.g. `[CLS]`, `[SEP]`, `[PAD]`, `<|endoftext|>`, `[BOS]`/`[EOS]`. |
| **`attention_mask`** | Marks real-content positions (`1`) vs. padding positions (`0`) so padding never affects real predictions. |
| **`token_type_ids`** | Marks which of two packed segments (e.g. question vs. context) each token belongs to. |
| **Padding** | Extending shorter sequences in a batch with a pad token so all rows share one rectangular tensor shape. |
| **Truncation** | Cutting sequences longer than `max_length` down to that length before (or instead of) padding. |
| **Chat template** | A per-model format (role markers + special tokens) that `apply_chat_template` builds from a list of `{role, content}` messages. |
| **Tokenizer/model mismatch** | Using a tokenizer from a different checkpoint than the model — ids decode without error but are semantically meaningless. |

---
